# 02. 최종 Predictive LSTM 학습 · 예측 · 영향요인 · 실패조건 · 안정 공정조건 분석

이 노트북은 `01_model_comparison.ipynb`에서 최종 선정된 **Predictive LSTM**을 독립적으로 재현한다.

- 입력: `data/Welding Data Set_01.xlsx`
- Fit(학습 구간): 2020-03-24 ~ 2020-03-27
- Calibration(임계값 보정구간): 2020-03-30 ~ 2020-03-31
- Holdout(후속 평가구간): 2020-04-02, 2020-04-03, 2020-04-07
- 모델 설정은 고정하며 **Early Stopping만 적용**한다.
- 개별 제품 정상/불량 라벨이 없으므로, 생산기록별 결과는 `불량 확정`이 아니라 **불량위험 이상탐지**이다.
- 안정 공정조건은 실제 무불량 보증범위가 아니라 **모델 기준 저위험 후보범위**이다.


In [41]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

# 시스템에 설치된 폰트 이름 확인
# font_list = [font.name for font in fm.fontManager.ttflist]
# print(font_list)

plt.rcParams['font.family'] = 'NanumGothic'  
plt.rcParams['axes.unicode_minus'] = False


In [42]:

# 0. 패키지 / 설정

from pathlib import Path
import json
import hashlib
import random
import copy
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 가능한 한 재현성 강화
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

ROOT_DIR = Path.cwd()
DATA_DIR = ROOT_DIR / "data"
MODEL_DIR = ROOT_DIR / "models"
OUTPUT_DIR = ROOT_DIR / "outputs" / "model_02_final"
PRED_DIR = OUTPUT_DIR / "predictions"
METRIC_DIR = OUTPUT_DIR / "metrics"
ANALYSIS_DIR = OUTPUT_DIR / "analysis"
FIG_DIR = ANALYSIS_DIR / "figures"

for p in [DATA_DIR, MODEL_DIR, PRED_DIR, METRIC_DIR, ANALYSIS_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# 파일명이 다르면 여기만 수정 가능
DATA_PATH = DATA_DIR / "Welding Data Set_01.xlsx"

FEATURES = [
    "weld force(bar)",
    "weld current(kA)",
    "weld Voltage(v)",
    "weld time(ms)",
]

FEATURE_KR = {
    "weld force(bar)": "가압력",
    "weld current(kA)": "전류",
    "weld Voltage(v)": "전압",
    "weld time(ms)": "통전시간",
}

FIT_DATES = pd.to_datetime([
    "2020-03-24", "2020-03-25", "2020-03-26", "2020-03-27"
]).normalize()

CAL_DATES = pd.to_datetime([
    "2020-03-30", "2020-03-31"
]).normalize()

HOLDOUT_DATES = pd.to_datetime([
    "2020-04-02", "2020-04-03", "2020-04-07"
]).normalize()

# 01과 동일하게 고정
CFG = {
    "model_name": "Predictive_LSTM",
    "sequence_length": 10,
    "hidden_dim": 32,
    "num_layers": 1,
    "dropout": 0.2,
    "batch_size": 32,
    "learning_rate": 0.001,
    "max_epochs": 300,
    "patience": 10,
    "loss": "mse",
    "score_type": "mae",
    "scaler": "standard",
    "seed": SEED,
    "warmup_policy": "no_prediction",
}

print(json.dumps(CFG, indent=2, ensure_ascii=False))


DEVICE: cuda
{
  "model_name": "Predictive_LSTM",
  "sequence_length": 10,
  "hidden_dim": 32,
  "num_layers": 1,
  "dropout": 0.2,
  "batch_size": 32,
  "learning_rate": 0.001,
  "max_epochs": 300,
  "patience": 10,
  "loss": "mse",
  "score_type": "mae",
  "scaler": "standard",
  "seed": 42,
  "warmup_policy": "no_prediction"
}


In [43]:

# 1. 원본 Excel 로드 및 데이터 점검

def resolve_excel():
    candidates = [
        DATA_PATH,
        ROOT_DIR / "Welding Data Set_01.xlsx",
        ROOT_DIR / "Welding Data Set_01(6).xlsx",
    ]
    candidates += sorted(DATA_DIR.glob("Welding Data Set_01*.xlsx"))
    candidates += sorted(ROOT_DIR.glob("Welding Data Set_01*.xlsx"))

    seen = set()
    for p in candidates:
        p = Path(p)
        if p in seen:
            continue
        seen.add(p)
        if p.exists() and p.is_file() and not p.name.startswith("~$"):
            return p.resolve()
    raise FileNotFoundError(
        "Welding Data Set_01.xlsx를 data 폴더 또는 현재 폴더에 넣어주세요."
    )


def clean_headers(df):
    out = df.copy()
    out.columns = [" ".join(str(c).strip().split()) for c in out.columns]
    return out


INPUT_PATH = resolve_excel()
print("INPUT:", INPUT_PATH)
print("SHA256:", hashlib.sha256(INPUT_PATH.read_bytes()).hexdigest())

raw = clean_headers(pd.read_excel(INPUT_PATH, sheet_name="Raw data"))
result = clean_headers(pd.read_excel(INPUT_PATH, sheet_name="result"))

required_raw = ["idx", "Machine_Name", "Item No", "working time"] + FEATURES
required_result = ["working time", "defect"]

missing_raw = [c for c in required_raw if c not in raw.columns]
missing_result = [c for c in required_result if c not in result.columns]
if missing_raw:
    raise ValueError(f"Raw data 누락 컬럼: {missing_raw}")
if missing_result:
    raise ValueError(f"result 누락 컬럼: {missing_result}")

raw.insert(0, "source_row", np.arange(1, len(raw) + 1))
raw["working time"] = pd.to_datetime(raw["working time"], errors="raise")
raw["date"] = raw["working time"].dt.normalize()

result["working time"] = pd.to_datetime(result["working time"], errors="raise")
result["date"] = result["working time"].dt.normalize()
result["defect"] = pd.to_numeric(result["defect"], errors="raise")

for c in FEATURES:
    raw[c] = pd.to_numeric(raw[c], errors="raise")

# 같은 날짜 안에서 idx를 생산순서 근사치로 사용
raw = raw.sort_values(
    ["date", "Machine_Name", "Item No", "idx", "source_row"],
    kind="mergesort"
).reset_index(drop=True)

daily_actual = (
    result.groupby("date", as_index=False)["defect"]
    .sum()
    .rename(columns={"defect": "actual_defect_count"})
)

print("Raw shape   :", raw.shape)
print("Result shape:", result.shape)
print("\n날짜별 생산건수")
print(raw.groupby("date").size())
print("\n날짜별 기록된 실제 불량건수")
print(daily_actual)


INPUT: /home/kampuser/welder_source_code/Untitled Folder/Welding Data Set_01.xlsx
SHA256: d514d6aaa121630c04d7d51c97a56025e78e2c86868813f7722ba5db922c1f33
Raw shape   : (11939, 12)
Result shape: (23, 8)

날짜별 생산건수
date
2020-03-24    1200
2020-03-25    1352
2020-03-26    1000
2020-03-27    1648
2020-03-30    1470
2020-03-31    1800
2020-04-02    2000
2020-04-03     800
2020-04-07     669
dtype: int64

날짜별 기록된 실제 불량건수
        date  actual_defect_count
0 2020-03-24                    4
1 2020-03-25                    8
2 2020-03-26                    7
3 2020-03-30                    4
4 2020-03-31                    5
5 2020-04-02                    4
6 2020-04-03                    4
7 2020-04-07                    3


In [44]:

# 2. Fit / Calibration / Holdout 분할

fit_df = raw[raw["date"].isin(FIT_DATES)].copy()
cal_df = raw[raw["date"].isin(CAL_DATES)].copy()
holdout_df = raw[raw["date"].isin(HOLDOUT_DATES)].copy()

print("Fit        :", len(fit_df))
print("Calibration:", len(cal_df))
print("Holdout    :", len(holdout_df))

# 현재 제공 데이터 기준 안전 점검
assert len(fit_df) == 5200, f"Fit 예상 5200, 실제 {len(fit_df)}"
assert len(cal_df) == 3270, f"Calibration 예상 3270, 실제 {len(cal_df)}"
assert len(holdout_df) == 3469, f"Holdout 예상 3469, 실제 {len(holdout_df)}"

split_table = pd.DataFrame([
    ["Fit", fit_df["date"].min(), fit_df["date"].max(), len(fit_df), "모델 학습 / scaler fit"],
    ["Calibration", cal_df["date"].min(), cal_df["date"].max(), len(cal_df), "Early stopping / threshold 보정"],
    ["Holdout", holdout_df["date"].min(), holdout_df["date"].max(), len(holdout_df), "후속 성능 확인"],
], columns=["split", "start", "end", "rows", "role"])

display(split_table)
split_table.to_csv(METRIC_DIR / "data_split.csv", index=False, encoding="utf-8-sig")


Fit        : 5200
Calibration: 3270
Holdout    : 3469


,split,start,end,rows,role
0,Fit,2020-03-24,2020-03-27,5200,모델 학습 / scaler fit
1,Calibration,2020-03-30,2020-03-31,3270,Early stopping / threshold 보정
2,Holdout,2020-04-02,2020-04-07,3469,후속 성능 확인


In [45]:

# 3. StandardScaler - Fit 구간에만 적합

scaler = StandardScaler()
scaler.fit(fit_df[FEATURES].to_numpy(dtype=np.float32))

joblib.dump(scaler, MODEL_DIR / "predictive_lstm_scaler.pkl")

print("Scaler mean :", scaler.mean_)
print("Scaler scale:", scaler.scale_)


Scaler mean : [ 2.85314996 14.71647883  2.70459807 71.72605769]
Scaler scale: [1.5493051  0.10191004 0.02640451 0.63064362]


In [46]:

# 4. 시퀀스 생성 함수
# - 날짜 / 설비 / 품목 경계를 넘지 않음
# - 과거 10개 기록 -> 현재 1개 기록 예측

def build_sequences(df, scaler, seq_len):
    X_list, y_list, meta_rows = [], [], []

    # 원본 행 전체를 먼저 scale
    work = df.copy()
    scaled_all = scaler.transform(work[FEATURES].to_numpy(dtype=np.float32)).astype(np.float32)
    for j, c in enumerate(FEATURES):
        work[f"scaled__{c}"] = scaled_all[:, j]

    group_cols = ["date", "Machine_Name", "Item No"]

    for _, g in work.groupby(group_cols, sort=False, dropna=False):
        g = g.sort_values(["idx", "source_row"], kind="mergesort").reset_index(drop=True)
        arr = g[[f"scaled__{c}" for c in FEATURES]].to_numpy(dtype=np.float32)

        if len(g) <= seq_len:
            continue

        for i in range(seq_len, len(g)):
            X_list.append(arr[i-seq_len:i])
            y_list.append(arr[i])
            meta_rows.append(g.iloc[i][[
                "source_row", "idx", "Machine_Name", "Item No", "working time", "date", *FEATURES
            ]].to_dict())

    if not X_list:
        return (
            np.empty((0, seq_len, len(FEATURES)), dtype=np.float32),
            np.empty((0, len(FEATURES)), dtype=np.float32),
            pd.DataFrame()
        )

    return (
        np.stack(X_list).astype(np.float32),
        np.stack(y_list).astype(np.float32),
        pd.DataFrame(meta_rows)
    )

SEQ_LEN = CFG["sequence_length"]
X_fit, y_fit, meta_fit = build_sequences(fit_df, scaler, SEQ_LEN)
X_cal, y_cal, meta_cal = build_sequences(cal_df, scaler, SEQ_LEN)
X_hold, y_hold, meta_hold = build_sequences(holdout_df, scaler, SEQ_LEN)

print("Fit sequences        :", X_fit.shape, y_fit.shape)
print("Calibration sequences:", X_cal.shape, y_cal.shape)
print("Holdout sequences    :", X_hold.shape, y_hold.shape)
print("Holdout warmup rows  :", len(holdout_df) - len(meta_hold))


Fit sequences        : (5160, 10, 4) (5160, 4)
Calibration sequences: (3250, 10, 4) (3250, 4)
Holdout sequences    : (3439, 10, 4) (3439, 4)
Holdout warmup rows  : 30


In [47]:

# 5. Predictive LSTM 정의
# welder_test2와 같은 핵심 구조:
# LSTM(32) -> Dropout(0.2) -> Linear(4)

class PredictiveLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, dropout, output_dim):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.0,  # 1 layer에서는 LSTM 내부 dropout 사용 안 함
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        last = self.dropout(last)
        return self.fc(last)


model = PredictiveLSTM(
    input_dim=len(FEATURES),
    hidden_dim=CFG["hidden_dim"],
    num_layers=CFG["num_layers"],
    dropout=CFG["dropout"],
    output_dim=len(FEATURES),
).to(DEVICE)

print(model)


PredictiveLSTM(
  (lstm): LSTM(4, 32, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=32, out_features=4, bias=True)
)


In [48]:

# 6. 모델 학습 + Early Stopping
# - Train loss: MSE
# - Validation: Calibration MSE
# - 하이퍼파라미터 탐색 없음

def make_loader(X, y, batch_size, shuffle=False):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)


def eval_mse(model, X, y, batch_size=1024):
    model.eval()
    loader = make_loader(X, y, batch_size=batch_size, shuffle=False)
    losses = []
    crit = nn.MSELoss(reduction="none")
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            pred = model(xb)
            loss = crit(pred, yb).mean(dim=1)
            losses.append(loss.detach().cpu().numpy())
    return float(np.concatenate(losses).mean())


train_loader = make_loader(
    X_fit, y_fit,
    batch_size=CFG["batch_size"],
    shuffle=False,
)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=CFG["learning_rate"])

best_val = np.inf
best_epoch = 0
best_state = None
wait = 0
history = []

for epoch in range(1, CFG["max_epochs"] + 1):
    model.train()
    train_losses = []

    for xb, yb in train_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    train_loss = float(np.mean(train_losses))
    val_loss = eval_mse(model, X_cal, y_cal)
    history.append([epoch, train_loss, val_loss])

    print(f"Epoch {epoch:03d} | train={train_loss:.8f} | cal={val_loss:.8f}")

    if val_loss < best_val - 1e-10:
        best_val = val_loss
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= CFG["patience"]:
            print(f"Early stopping: epoch={epoch}, best_epoch={best_epoch}")
            break

model.load_state_dict(best_state)

history_df = pd.DataFrame(history, columns=["epoch", "train_mse", "calibration_mse"])
history_df.to_csv(METRIC_DIR / "training_history.csv", index=False, encoding="utf-8-sig")

print("BEST EPOCH:", best_epoch)
print("BEST CAL MSE:", best_val)


Epoch 001 | train=0.80057655 | cal=0.51401526
Epoch 002 | train=0.50881362 | cal=0.40235978
Epoch 003 | train=0.43451853 | cal=0.36836314
Epoch 004 | train=0.40855905 | cal=0.35440272
Epoch 005 | train=0.39285299 | cal=0.33076045
Epoch 006 | train=0.36268100 | cal=0.31015077
Epoch 007 | train=0.34232956 | cal=0.29558858
Epoch 008 | train=0.32608956 | cal=0.28473881
Epoch 009 | train=0.31604621 | cal=0.27245238
Epoch 010 | train=0.30126427 | cal=0.25899163
Epoch 011 | train=0.29277707 | cal=0.24715720
Epoch 012 | train=0.28188696 | cal=0.23959011
Epoch 013 | train=0.27510112 | cal=0.22639968
Epoch 014 | train=0.26070893 | cal=0.21805593
Epoch 015 | train=0.25273831 | cal=0.21337335
Epoch 016 | train=0.24822905 | cal=0.21716525
Epoch 017 | train=0.24650890 | cal=0.21594130
Epoch 018 | train=0.24372638 | cal=0.20507944
Epoch 019 | train=0.23382032 | cal=0.19564795
Epoch 020 | train=0.22737243 | cal=0.19215353
Epoch 021 | train=0.22361359 | cal=0.19423185
Epoch 022 | train=0.21910947 | cal

In [49]:

# 7. 예측 / 이상점수 생성
# anomaly_score = scaled 4개 변수 절대예측오차의 평균(MAE)

def predict_arrays(model, X, batch_size=1024):
    model.eval()
    preds = []
    loader = DataLoader(
        torch.tensor(X, dtype=torch.float32),
        batch_size=batch_size,
        shuffle=False,
    )
    with torch.no_grad():
        for xb in loader:
            pred = model(xb.to(DEVICE))
            preds.append(pred.detach().cpu().numpy())
    return np.concatenate(preds, axis=0)


def make_scored_frame(model, X, y, meta, scaler):
    pred_scaled = predict_arrays(model, X)
    abs_err_scaled = np.abs(y - pred_scaled)
    score = abs_err_scaled.mean(axis=1)

    actual_original = scaler.inverse_transform(y)
    pred_original = scaler.inverse_transform(pred_scaled)
    abs_err_original = np.abs(actual_original - pred_original)

    out = meta.reset_index(drop=True).copy()
    out["anomaly_score"] = score

    for j, c in enumerate(FEATURES):
        safe = c.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")
        out[f"pred__{c}"] = pred_original[:, j]
        out[f"abs_err_scaled__{c}"] = abs_err_scaled[:, j]
        out[f"abs_err_original__{c}"] = abs_err_original[:, j]

    return out


fit_score_df = make_scored_frame(model, X_fit, y_fit, meta_fit, scaler)
cal_score_df = make_scored_frame(model, X_cal, y_cal, meta_cal, scaler)
hold_score_df = make_scored_frame(model, X_hold, y_hold, meta_hold, scaler)

print("Fit score rows:", len(fit_score_df))
print("Cal score rows:", len(cal_score_df))
print("Hold score rows:", len(hold_score_df))


Fit score rows: 5160
Cal score rows: 3250
Hold score rows: 3439


In [50]:

# 8. Calibration 일별 집계에 맞춘 Threshold 보정
# - Threshold 후보는 Fit score quantile로 생성
# - Calibration Daily MAE -> RMSE -> 총량오차 순으로 선택

def daily_eval(scored_df, threshold, actual_daily, eval_dates):
    temp = scored_df.copy()
    temp["anomaly_flag"] = (temp["anomaly_score"] >= threshold).astype(int)

    pred_daily = (
        temp.groupby("date", as_index=False)["anomaly_flag"]
        .sum()
        .rename(columns={"anomaly_flag": "predicted_anomaly_count"})
    )

    base = pd.DataFrame({"date": pd.to_datetime(eval_dates).normalize()})
    out = (
        base.merge(actual_daily, on="date", how="left")
        .merge(pred_daily, on="date", how="left")
    )
    out["predicted_anomaly_count"] = out["predicted_anomaly_count"].fillna(0).astype(int)
    out["error"] = out["predicted_anomaly_count"] - out["actual_defect_count"]
    out["absolute_error"] = out["error"].abs()
    out["squared_error"] = out["error"] ** 2
    return out


qs = np.linspace(0.9900, 0.9999, 500)
threshold_rows = []

for q in qs:
    th = float(np.quantile(fit_score_df["anomaly_score"], q))
    d = daily_eval(cal_score_df, th, daily_actual, CAL_DATES)

    mae = float(d["absolute_error"].mean())
    rmse = float(np.sqrt(d["squared_error"].mean()))
    bias = float(d["error"].mean())
    actual_total = float(d["actual_defect_count"].sum())
    pred_total = float(d["predicted_anomaly_count"].sum())
    total_error = abs(pred_total - actual_total)

    threshold_rows.append({
        "quantile": q,
        "threshold": th,
        "daily_mae": mae,
        "daily_rmse": rmse,
        "daily_bias": bias,
        "actual_total": actual_total,
        "predicted_total": pred_total,
        "absolute_total_count_error": total_error,
    })

threshold_search = pd.DataFrame(threshold_rows).sort_values(
    ["daily_mae", "daily_rmse", "absolute_total_count_error", "threshold"],
    kind="mergesort"
).reset_index(drop=True)

BEST_THRESHOLD = float(threshold_search.iloc[0]["threshold"])
BEST_Q = float(threshold_search.iloc[0]["quantile"])

print("BEST Q        :", BEST_Q)
print("BEST THRESHOLD:", BEST_THRESHOLD)
display(threshold_search.head(10))

threshold_search.to_csv(
    METRIC_DIR / "threshold_calibration_search.csv",
    index=False,
    encoding="utf-8-sig"
)


BEST Q        : 0.998074749498998
BEST THRESHOLD: 0.8339118304542418


,quantile,threshold,daily_mae,daily_rmse,daily_bias,actual_total,predicted_total,absolute_total_count_error
0,0.998075,0.833912,1.0,1.414214,-1.0,9.0,7.0,2.0
1,0.998095,0.834194,1.0,1.414214,-1.0,9.0,7.0,2.0
2,0.998114,0.834476,1.0,1.414214,-1.0,9.0,7.0,2.0
3,0.998134,0.834759,1.0,1.414214,-1.0,9.0,7.0,2.0
4,0.998154,0.835041,1.0,1.414214,-1.0,9.0,7.0,2.0
5,0.998174,0.835323,1.0,1.414214,-1.0,9.0,7.0,2.0
6,0.998194,0.835606,1.0,1.414214,-1.0,9.0,7.0,2.0
7,0.998214,0.835888,1.0,1.414214,-1.0,9.0,7.0,2.0
8,0.998233,0.836170,1.0,1.414214,-1.0,9.0,7.0,2.0
9,0.998253,0.836453,1.0,1.414214,-1.0,9.0,7.0,2.0


In [51]:

# 9. Calibration / Holdout 최종 평가

for frame in [fit_score_df, cal_score_df, hold_score_df]:
    frame["threshold"] = BEST_THRESHOLD
    frame["anomaly_flag"] = (frame["anomaly_score"] >= BEST_THRESHOLD).astype(int)
    frame["score_ratio_to_threshold"] = frame["anomaly_score"] / BEST_THRESHOLD

cal_daily = daily_eval(cal_score_df, BEST_THRESHOLD, daily_actual, CAL_DATES)
hold_daily = daily_eval(hold_score_df, BEST_THRESHOLD, daily_actual, HOLDOUT_DATES)


def metric_summary(name, daily):
    return {
        "split": name,
        "daily_mae": float(daily["absolute_error"].mean()),
        "daily_rmse": float(np.sqrt(daily["squared_error"].mean())),
        "daily_bias": float(daily["error"].mean()),
        "actual_total": int(daily["actual_defect_count"].sum()),
        "predicted_total": int(daily["predicted_anomaly_count"].sum()),
    }

final_metrics = pd.DataFrame([
    metric_summary("Calibration", cal_daily),
    metric_summary("Holdout", hold_daily),
])

print("\nCALIBRATION")
display(cal_daily)
print("\nHOLDOUT")
display(hold_daily)
print("\nFINAL METRICS")
display(final_metrics)

cal_daily.to_csv(METRIC_DIR / "calibration_daily_evaluation.csv", index=False, encoding="utf-8-sig")
hold_daily.to_csv(METRIC_DIR / "holdout_daily_evaluation.csv", index=False, encoding="utf-8-sig")
final_metrics.to_csv(METRIC_DIR / "final_model_metrics.csv", index=False, encoding="utf-8-sig")



CALIBRATION


,date,actual_defect_count,predicted_anomaly_count,error,absolute_error,squared_error
0,2020-03-30,4,2,-2,2,4
1,2020-03-31,5,5,0,0,0



HOLDOUT


,date,actual_defect_count,predicted_anomaly_count,error,absolute_error,squared_error
0,2020-04-02,4,4,0,0,0
1,2020-04-03,4,4,0,0,0
2,2020-04-07,3,3,0,0,0



FINAL METRICS


,split,daily_mae,daily_rmse,daily_bias,actual_total,predicted_total
0,Calibration,1.0,1.414214,-1.0,9,7
1,Holdout,0.0,0.000000,0.0,11,11


In [52]:

# 10. 모든 Holdout 행에 결과 연결
# - 각 날짜 첫 10개는 시퀀스 부족으로 score 없음
# - 이를 정상으로 강제 판정하지 않고 'warmup_no_prediction'으로 명시

score_cols = [
    "source_row", "anomaly_score", "threshold", "anomaly_flag", "score_ratio_to_threshold",
]
for c in FEATURES:
    score_cols += [
        f"pred__{c}",
        f"abs_err_scaled__{c}",
        f"abs_err_original__{c}",
    ]

holdout_all = holdout_df.merge(
    hold_score_df[score_cols],
    on="source_row",
    how="left",
    validate="one_to_one",
)

holdout_all["score_source"] = np.where(
    holdout_all["anomaly_score"].notna(),
    "Predictive_LSTM",
    "warmup_no_prediction",
)

# nullable integer: warmup은 NA 유지
holdout_all["anomaly_flag"] = holdout_all["anomaly_flag"].astype("Int64")

# scored row 안에서의 상대순위. 불량확률 아님.
valid_scores = holdout_all["anomaly_score"].dropna()
holdout_all["risk_percentile"] = np.nan
if len(valid_scores):
    holdout_all.loc[holdout_all["anomaly_score"].notna(), "risk_percentile"] = (
        holdout_all.loc[holdout_all["anomaly_score"].notna(), "anomaly_score"]
        .rank(method="average", pct=True)
    )

holdout_all.to_csv(
    PRED_DIR / "final_holdout_predictions_all_rows.csv",
    index=False,
    encoding="utf-8-sig",
)

hold_score_df.to_csv(
    PRED_DIR / "final_holdout_predictions_scored_rows.csv",
    index=False,
    encoding="utf-8-sig",
)

print("전체 Holdout:", len(holdout_all))
print("실제 LSTM score 생성:", holdout_all["anomaly_score"].notna().sum())
print("Warmup 미예측:", holdout_all["anomaly_score"].isna().sum())
display(holdout_all.head(15))


전체 Holdout: 3469
실제 LSTM score 생성: 3439
Warmup 미예측: 30


,source_row,idx,Machine_Name,Item No,working time,Thickness 1(mm),Thickness 2(mm),weld force(bar),weld current(kA),weld Voltage(v),...,abs_err_scaled__weld current(kA),abs_err_original__weld current(kA),pred__weld Voltage(v),abs_err_scaled__weld Voltage(v),abs_err_original__weld Voltage(v),pred__weld time(ms),abs_err_scaled__weld time(ms),abs_err_original__weld time(ms),score_source,risk_percentile
0,8471,1,Spot-01,65235-25800,2020-04-02,0.7,0.7,2.36,14.74,2.702,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,warmup_no_prediction,NaN
1,8472,2,Spot-01,65235-25800,2020-04-02,0.7,0.7,2.35,14.73,2.708,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,warmup_no_prediction,NaN
2,8473,3,Spot-01,65235-25800,2020-04-02,0.7,0.7,2.35,14.73,2.708,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,warmup_no_prediction,NaN
3,8474,4,Spot-01,65235-25800,2020-04-02,0.7,0.7,2.35,14.73,2.705,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,warmup_no_prediction,NaN
4,8475,5,Spot-01,65235-25800,2020-04-02,0.7,0.7,2.35,14.73,2.705,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,warmup_no_prediction,NaN
5,8476,6,Spot-01,65235-25800,2020-04-02,0.7,0.7,2.33,14.75,2.701,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,warmup_no_prediction,NaN
6,8477,7,Spot-01,65235-25800,2020-04-02,0.7,0.7,2.33,14.75,2.701,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,warmup_no_prediction,NaN
7,8478,8,Spot-01,65235-25800,2020-04-02,0.7,0.7,2.33,14.76,2.699,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,warmup_no_prediction,NaN
8,8479,9,Spot-01,65235-25800,2020-04-02,0.7,0.7,2.33,14.76,2.699,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,warmup_no_prediction,NaN
9,8480,10,Spot-01,65235-25800,2020-04-02,0.7,0.7,2.33,14.72,2.698,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,warmup_no_prediction,NaN


In [53]:

# 11. 모델 / Scaler / Threshold / 설정 저장 + 재로딩 검증

MODEL_PATH = MODEL_DIR / "predictive_lstm_final.pt"
SCALER_PATH = MODEL_DIR / "predictive_lstm_scaler.pkl"
CONFIG_PATH = MODEL_DIR / "predictive_lstm_config.json"
THRESHOLD_PATH = MODEL_DIR / "predictive_lstm_threshold.json"

save_obj = {
    "state_dict": model.state_dict(),
    "input_dim": len(FEATURES),
    "hidden_dim": CFG["hidden_dim"],
    "num_layers": CFG["num_layers"],
    "dropout": CFG["dropout"],
    "output_dim": len(FEATURES),
    "best_epoch": best_epoch,
}
torch.save(save_obj, MODEL_PATH)
joblib.dump(scaler, SCALER_PATH)

config_to_save = dict(CFG)
config_to_save.update({
    "features": FEATURES,
    "fit_dates": [str(d.date()) for d in FIT_DATES],
    "calibration_dates": [str(d.date()) for d in CAL_DATES],
    "holdout_dates": [str(d.date()) for d in HOLDOUT_DATES],
    "best_epoch": int(best_epoch),
    "best_calibration_mse": float(best_val),
})
CONFIG_PATH.write_text(json.dumps(config_to_save, indent=2, ensure_ascii=False), encoding="utf-8")
THRESHOLD_PATH.write_text(json.dumps({
    "threshold": BEST_THRESHOLD,
    "fit_score_quantile": BEST_Q,
    "selection_metric": "Calibration Daily MAE -> RMSE -> total count error",
}, indent=2, ensure_ascii=False), encoding="utf-8")

# 재로딩 검증
loaded = torch.load(MODEL_PATH, map_location=DEVICE)
reloaded_model = PredictiveLSTM(
    input_dim=loaded["input_dim"],
    hidden_dim=loaded["hidden_dim"],
    num_layers=loaded["num_layers"],
    dropout=loaded["dropout"],
    output_dim=loaded["output_dim"],
).to(DEVICE)
reloaded_model.load_state_dict(loaded["state_dict"])
reloaded_model.eval()

n_check = min(200, len(X_hold))
orig_pred = predict_arrays(model, X_hold[:n_check])
reload_pred = predict_arrays(reloaded_model, X_hold[:n_check])
max_diff = float(np.max(np.abs(orig_pred - reload_pred)))
print("Reload max prediction diff:", max_diff)
assert np.allclose(orig_pred, reload_pred, atol=1e-6), "재로딩 모델 예측 불일치"

print("MODEL SAVED:", MODEL_PATH)
print("SCALER SAVED:", SCALER_PATH)
print("CONFIG SAVED:", CONFIG_PATH)
print("THRESHOLD SAVED:", THRESHOLD_PATH)


Reload max prediction diff: 0.0
MODEL SAVED: /home/kampuser/welder_source_code/Untitled Folder/models/predictive_lstm_final.pt
SCALER SAVED: /home/kampuser/welder_source_code/Untitled Folder/models/predictive_lstm_scaler.pkl
CONFIG SAVED: /home/kampuser/welder_source_code/Untitled Folder/models/predictive_lstm_config.json
THRESHOLD SAVED: /home/kampuser/welder_source_code/Untitled Folder/models/predictive_lstm_threshold.json


## 12. 날짜별 예측 실패 분석

개별 제품 라벨이 없으므로 `FN/FP`라고 부르지 않는다. 날짜별로 실제 집계 불량건수보다 이상탐지가 적으면 **과소이상탐지**, 많으면 **과대이상탐지**로 표현한다.


In [54]:

# 12. 날짜별 예측 실패조건 분석

failure_daily = hold_daily.copy()
failure_daily["status"] = np.select(
    [failure_daily["error"] < 0, failure_daily["error"] > 0],
    ["과소이상탐지", "과대이상탐지"],
    default="건수일치",
)

# threshold 바로 아래의 고위험 비이상탐지 기록 수
# score >= 0.90 * threshold 이면서 flag=0
near = hold_score_df[
    (hold_score_df["anomaly_flag"] == 0) &
    (hold_score_df["anomaly_score"] >= 0.90 * BEST_THRESHOLD)
].copy()

near_daily = (
    near.groupby("date", as_index=False)
    .agg(
        near_threshold_nonalert_count=("source_row", "size"),
        max_nonalert_score=("anomaly_score", "max"),
        mean_nonalert_score=("anomaly_score", "mean"),
    )
)

warmup_daily = (
    holdout_all.assign(is_warmup=holdout_all["anomaly_score"].isna().astype(int))
    .groupby("date", as_index=False)["is_warmup"].sum()
    .rename(columns={"is_warmup": "warmup_no_prediction_count"})
)

failure_daily = (
    failure_daily
    .merge(near_daily, on="date", how="left")
    .merge(warmup_daily, on="date", how="left")
)

failure_daily["near_threshold_nonalert_count"] = failure_daily["near_threshold_nonalert_count"].fillna(0).astype(int)
failure_daily["warmup_no_prediction_count"] = failure_daily["warmup_no_prediction_count"].fillna(0).astype(int)

failure_daily["interpretation"] = np.where(
    failure_daily["status"].eq("과소이상탐지"),
    "실제 집계보다 모델 이상탐지 수가 적음. threshold 직하위 및 warmup 기록 우선 재검토",
    np.where(
        failure_daily["status"].eq("과대이상탐지"),
        "실제 집계보다 모델 이상탐지 수가 많음. 공정변화 민감도 또는 불필요 이상탐지 가능성 점검",
        "일별 건수는 일치하지만 동일 제품을 맞혔다는 의미는 아님",
    )
)

failure_daily.to_csv(
    ANALYSIS_DIR / "daily_failure_analysis.csv",
    index=False,
    encoding="utf-8-sig",
)

# 날짜별 threshold 직하위 TOP 10 기록
near_top10 = (
    near.sort_values(["date", "anomaly_score"], ascending=[True, False])
    .groupby("date", group_keys=False)
    .head(10)
)
near_top10.to_csv(
    ANALYSIS_DIR / "near_threshold_nonalert_top10_by_day.csv",
    index=False,
    encoding="utf-8-sig",
)

display(failure_daily)


,date,actual_defect_count,predicted_anomaly_count,error,absolute_error,squared_error,status,near_threshold_nonalert_count,max_nonalert_score,mean_nonalert_score,warmup_no_prediction_count,interpretation
0,2020-04-02,4,4,0,0,0,건수일치,1,0.770827,0.770827,10,일별 건수는 일치하지만 동일 제품을 맞혔다는 의미는 아님
1,2020-04-03,4,4,0,0,0,건수일치,3,0.833725,0.801383,10,일별 건수는 일치하지만 동일 제품을 맞혔다는 의미는 아님
2,2020-04-07,3,3,0,0,0,건수일치,1,0.828783,0.828783,10,일별 건수는 일치하지만 동일 제품을 맞혔다는 의미는 아님


## 13. 주요 공정변수 분석

Predictive LSTM은 다음 4개 공정값을 예측하므로, 이상점수를 변수별 절대예측오차로 분해할 수 있다. 이는 **모델 이상탐지 점수에 대한 기여도**이며 실제 불량의 인과적 중요도를 의미하지 않는다.


In [55]:

# 13. 변수별 예측오차 기여도

def feature_contribution_table(scored, scope_name):
    target = scored.copy()
    alert = target[target["anomaly_flag"] == 1].copy()

    rows = []
    total_alert_err = sum(alert[f"abs_err_scaled__{c}"].sum() for c in FEATURES)
    total_all_err = sum(target[f"abs_err_scaled__{c}"].sum() for c in FEATURES)

    for c in FEATURES:
        alert_sum = float(alert[f"abs_err_scaled__{c}"].sum())
        all_sum = float(target[f"abs_err_scaled__{c}"].sum())
        rows.append({
            "scope": scope_name,
            "feature": c,
            "feature_kr": FEATURE_KR[c],
            "alert_error_sum_scaled": alert_sum,
            "alert_contribution_pct": 100.0 * alert_sum / total_alert_err if total_alert_err > 0 else np.nan,
            "all_error_sum_scaled": all_sum,
            "all_contribution_pct": 100.0 * all_sum / total_all_err if total_all_err > 0 else np.nan,
            "mean_abs_error_original_alert": float(alert[f"abs_err_original__{c}"].mean()) if len(alert) else np.nan,
            "mean_abs_error_original_all": float(target[f"abs_err_original__{c}"].mean()),
        })

    return pd.DataFrame(rows).sort_values("alert_contribution_pct", ascending=False)

feature_contrib = feature_contribution_table(hold_score_df, "Holdout")
feature_contrib.to_csv(
    ANALYSIS_DIR / "feature_prediction_error_contribution.csv",
    index=False,
    encoding="utf-8-sig",
)

display(feature_contrib)


,scope,feature,feature_kr,alert_error_sum_scaled,alert_contribution_pct,all_error_sum_scaled,all_contribution_pct,mean_abs_error_original_alert,mean_abs_error_original_all
3,Holdout,weld time(ms),통전시간,23.512629,52.080477,741.132202,41.896199,1.348007,0.135909
2,Holdout,weld Voltage(v),전압,9.739601,21.573218,337.040771,19.052913,0.023379,0.002588
1,Holdout,weld current(kA),전류,6.125739,13.568512,529.072266,29.908452,0.056752,0.015678
0,Holdout,weld force(bar),가압력,5.768754,12.777792,161.727188,9.142437,0.812506,0.072860


In [56]:

# 14. 공정변수 값 구간별 이상탐지율
# - 이상탐지율은 실제 불량률이 아니라 모델 이상탐지 비율
# - 구간 경계는 Fit+Calibration 개발구간 원자료 분위수로 정의

dev_raw = pd.concat([fit_df, cal_df], ignore_index=True)


def unique_quantile_edges(series, n_bins=5):
    q = np.linspace(0, 1, n_bins + 1)
    edges = np.unique(np.quantile(series.astype(float), q))
    if len(edges) < 3:
        # 값 종류가 너무 적으면 unique mid 방식 대신 실제 unique를 경계로 사용
        vals = np.unique(series.astype(float))
        if len(vals) == 1:
            return np.array([vals[0] - 1e-9, vals[0] + 1e-9])
        edges = np.r_[vals[0] - 1e-9, (vals[:-1] + vals[1:]) / 2, vals[-1] + 1e-9]
    edges[0] = -np.inf
    edges[-1] = np.inf
    return edges


feature_bin_rows = []
feature_bin_tables = {}

for c in FEATURES:
    edges = unique_quantile_edges(dev_raw[c], n_bins=5)
    temp = hold_score_df[["source_row", "date", c, "anomaly_score", "anomaly_flag"]].copy()
    temp["bin"] = pd.cut(temp[c], bins=edges, include_lowest=True, duplicates="drop")

    tab = (
        temp.groupby("bin", observed=True)
        .agg(
            records=("source_row", "size"),
            alerts=("anomaly_flag", "sum"),
            mean_anomaly_score=("anomaly_score", "mean"),
            feature_mean=(c, "mean"),
        )
        .reset_index()
    )
    tab["alarm_rate"] = tab["alerts"] / tab["records"]
    tab.insert(0, "feature", c)
    tab.insert(1, "feature_kr", FEATURE_KR[c])
    tab["bin"] = tab["bin"].astype(str)

    feature_bin_tables[c] = tab
    feature_bin_rows.append(tab)

feature_alarm_bins = pd.concat(feature_bin_rows, ignore_index=True)
feature_alarm_bins.to_csv(
    ANALYSIS_DIR / "feature_alarm_rate_by_bin.csv",
    index=False,
    encoding="utf-8-sig",
)

display(feature_alarm_bins)


,feature,feature_kr,bin,records,alerts,mean_anomaly_score,feature_mean,alarm_rate
0,weld force(bar),가압력,"(-inf, 2.31]",887,1,0.113026,2.295344,0.001127
1,weld force(bar),가압력,"(2.31, 2.33]",672,5,0.112704,2.325119,0.007440
2,weld force(bar),가압력,"(2.33, 2.35]",583,0,0.106097,2.346141,0.000000
3,weld force(bar),가압력,"(2.35, 2.38]",830,0,0.124304,2.368590,0.000000
4,weld force(bar),가압력,"(2.38, inf]",467,5,0.216756,5.258158,0.010707
5,weld current(kA),전류,"(-inf, 14.6]",828,4,0.129546,14.574928,0.004831
6,weld current(kA),전류,"(14.6, 14.72]",847,3,0.118533,14.684404,0.003542
7,weld current(kA),전류,"(14.72, 14.74]",752,2,0.116554,14.735027,0.002660
8,weld current(kA),전류,"(14.74, 14.77]",511,0,0.107882,14.756751,0.000000
9,weld current(kA),전류,"(14.77, inf]",501,2,0.183245,14.856487,0.003992


## 15. 공정변수 간 상호작용

두 공정변수 조합별로 Holdout의 생산건수, 모델 이상탐지건수, 모델 이상탐지율, 평균 이상점수를 계산한다. 실제 개별 불량 라벨이 없으므로 상호작용 결과 역시 **불량 원인**이 아니라 **모델 이상탐지 집중 구간**으로 해석한다.


In [57]:

# 15. 두 변수 조합별 이상탐지 집중 분석

from itertools import combinations

interaction_tables = []

for f1, f2 in combinations(FEATURES, 2):
    e1 = unique_quantile_edges(dev_raw[f1], n_bins=4)
    e2 = unique_quantile_edges(dev_raw[f2], n_bins=4)

    temp = hold_score_df[[
        "source_row", "date", f1, f2, "anomaly_score", "anomaly_flag"
    ]].copy()
    temp["bin_1"] = pd.cut(temp[f1], bins=e1, include_lowest=True, duplicates="drop")
    temp["bin_2"] = pd.cut(temp[f2], bins=e2, include_lowest=True, duplicates="drop")

    tab = (
        temp.groupby(["bin_1", "bin_2"], observed=True)
        .agg(
            records=("source_row", "size"),
            alerts=("anomaly_flag", "sum"),
            mean_anomaly_score=("anomaly_score", "mean"),
        )
        .reset_index()
    )
    tab["alarm_rate"] = tab["alerts"] / tab["records"]
    tab["feature_1"] = f1
    tab["feature_2"] = f2
    tab["feature_1_kr"] = FEATURE_KR[f1]
    tab["feature_2_kr"] = FEATURE_KR[f2]
    tab["bin_1"] = tab["bin_1"].astype(str)
    tab["bin_2"] = tab["bin_2"].astype(str)

    interaction_tables.append(tab)

pairwise_interactions = pd.concat(interaction_tables, ignore_index=True)
pairwise_interactions.to_csv(
    ANALYSIS_DIR / "pairwise_interactions.csv",
    index=False,
    encoding="utf-8-sig",
)

# 표본 30건 이상인 조합 중 이상탐지율 상위
interaction_top = (
    pairwise_interactions[pairwise_interactions["records"] >= 30]
    .sort_values(["alarm_rate", "alerts", "records"], ascending=[False, False, False])
    .head(30)
)
interaction_top.to_csv(
    ANALYSIS_DIR / "pairwise_interactions_top30.csv",
    index=False,
    encoding="utf-8-sig",
)

display(interaction_top)


,bin_1,bin_2,records,alerts,mean_anomaly_score,alarm_rate,feature_1,feature_2,feature_1_kr,feature_2_kr
30,"(2.37, inf]","(2.702, 2.706]",93,2,0.176055,0.021505,weld force(bar),weld Voltage(v),가압력,전압
71,"(14.76, inf]","(72.0, inf]",48,1,0.170790,0.020833,weld current(kA),weld time(ms),전류,통전시간
74,"(-inf, 2.699]","(72.0, inf]",97,2,0.142945,0.020619,weld Voltage(v),weld time(ms),전압,통전시간
4,"(2.31, 2.34]","(-inf, 14.62]",100,2,0.118563,0.020000,weld force(bar),weld current(kA),가압력,전류
28,"(2.37, inf]","(-inf, 2.699]",156,3,0.218129,0.019231,weld force(bar),weld Voltage(v),가압력,전압
43,"(2.37, inf]","(72.0, inf]",110,2,0.157524,0.018182,weld force(bar),weld time(ms),가압력,통전시간
14,"(2.37, inf]","(14.73, 14.76]",63,1,0.258521,0.015873,weld force(bar),weld current(kA),가압력,전류
35,"(2.31, 2.34]","(-inf, 71.0]",319,5,0.131859,0.015674,weld force(bar),weld time(ms),가압력,통전시간
68,"(14.73, 14.76]","(72.0, inf]",70,1,0.128744,0.014286,weld current(kA),weld time(ms),전류,통전시간
21,"(2.31, 2.34]","(2.699, 2.702]",293,4,0.118455,0.013652,weld force(bar),weld Voltage(v),가압력,전압


## 16. 안정 공정조건 후보

개발구간(Fit+Calibration)의 모델 점수 하위 50%를 `low-risk pool`로 정의하고, 각 변수의 10~90% 구간을 안정 공정조건 **후보**로 제안한다. Holdout은 후보 범위 설정에 사용하지 않고 사후 확인에만 사용한다.


In [58]:

# 16. 모델 기반 저위험 안정 공정조건 후보범위 도출

dev_scored = pd.concat([
    fit_score_df.assign(split="Fit"),
    cal_score_df.assign(split="Calibration"),
], ignore_index=True)

low_risk_cut = float(dev_scored["anomaly_score"].quantile(0.50))
low_risk_pool = dev_scored[dev_scored["anomaly_score"] <= low_risk_cut].copy()

stable_rows = []
for c in FEATURES:
    lo = float(low_risk_pool[c].quantile(0.10))
    hi = float(low_risk_pool[c].quantile(0.90))
    stable_rows.append({
        "feature": c,
        "feature_kr": FEATURE_KR[c],
        "lower": lo,
        "upper": hi,
        "derivation": "development score bottom 50%, feature 10-90%",
    })

stable_ranges = pd.DataFrame(stable_rows)
stable_ranges.to_csv(
    ANALYSIS_DIR / "stable_condition_candidate_ranges.csv",
    index=False,
    encoding="utf-8-sig",
)

# Holdout scored rows에서 범위 내부/외부 모델 이상탐지율 확인
stable_mask = np.ones(len(hold_score_df), dtype=bool)
for _, r in stable_ranges.iterrows():
    stable_mask &= hold_score_df[r["feature"]].between(r["lower"], r["upper"], inclusive="both").to_numpy()

stable_eval_df = hold_score_df.copy()
stable_eval_df["inside_stable_candidate"] = stable_mask

stable_eval = (
    stable_eval_df.groupby("inside_stable_candidate")
    .agg(
        records=("source_row", "size"),
        alerts=("anomaly_flag", "sum"),
        mean_anomaly_score=("anomaly_score", "mean"),
    )
    .reset_index()
)
stable_eval["alarm_rate"] = stable_eval["alerts"] / stable_eval["records"]
stable_eval["condition"] = np.where(
    stable_eval["inside_stable_candidate"],
    "4개 후보범위 모두 충족",
    "하나 이상 후보범위 이탈",
)

stable_eval.to_csv(
    ANALYSIS_DIR / "stable_condition_holdout_evaluation.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Low-risk score cutoff:", low_risk_cut)
display(stable_ranges)
display(stable_eval[["condition", "records", "alerts", "alarm_rate", "mean_anomaly_score"]])


Low-risk score cutoff: 0.09833672642707825


,feature,feature_kr,lower,upper,derivation
0,weld force(bar),가압력,2.290,2.380,"development score bottom 50%, feature 10-90%"
1,weld current(kA),전류,14.580,14.770,"development score bottom 50%, feature 10-90%"
2,weld Voltage(v),전압,2.697,2.708,"development score bottom 50%, feature 10-90%"
3,weld time(ms),통전시간,71.000,72.160,"development score bottom 50%, feature 10-90%"


,condition,records,alerts,alarm_rate,mean_anomaly_score
0,하나 이상 후보범위 이탈,1720,9,0.005233,0.150181
1,4개 후보범위 모두 충족,1719,2,0.001163,0.106999


In [59]:

# 17. 그래프 자동 저장

# 17-1 날짜별 실제 불량 vs 모델 이상탐지
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(hold_daily))
w = 0.36
ax.bar(x - w/2, hold_daily["actual_defect_count"], width=w, label="Actual defects")
ax.bar(x + w/2, hold_daily["predicted_anomaly_count"], width=w, label="Model alerts")
ax.set_xticks(x)
ax.set_xticklabels([pd.Timestamp(d).strftime("%m-%d") for d in hold_daily["date"]])
ax.set_ylabel("Count")
ax.set_title("Holdout: actual defect count vs model alert count")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "daily_actual_vs_model_alerts.png", dpi=160, bbox_inches="tight")
plt.close(fig)

# 17-2 변수별 이상탐지점수 기여도
fig, ax = plt.subplots(figsize=(7, 4.5))
plot_df = feature_contrib.sort_values("alert_contribution_pct", ascending=True)
ax.barh(plot_df["feature_kr"], plot_df["alert_contribution_pct"])
ax.set_xlabel("Contribution to alert-row prediction error (%)")
ax.set_title("Feature contribution to Predictive LSTM alert score")
fig.tight_layout()
fig.savefig(FIG_DIR / "feature_error_contribution.png", dpi=160, bbox_inches="tight")
plt.close(fig)

# 17-3 안정 후보 범위 내부/외부 이상탐지율
fig, ax = plt.subplots(figsize=(7, 4.5))
se = stable_eval.copy()
ax.bar(se["condition"], se["alarm_rate"] * 100)
ax.set_ylabel("Model alarm rate (%)")
ax.set_title("Alarm rate inside/outside stable-condition candidate")
ax.tick_params(axis="x", rotation=15)
fig.tight_layout()
fig.savefig(FIG_DIR / "stable_candidate_alarm_rate.png", dpi=160, bbox_inches="tight")
plt.close(fig)

# 17-4 변수별 구간 이상탐지율
for c, tab in feature_bin_tables.items():
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(tab["bin"], tab["alarm_rate"] * 100)
    ax.set_ylabel("Model alarm rate (%)")
    ax.set_title(f"{FEATURE_KR[c]} bin-wise model alarm rate")
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    safe_name = c.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")
    fig.savefig(FIG_DIR / f"feature_bin_alarm_rate__{safe_name}.png", dpi=160, bbox_inches="tight")
    plt.close(fig)

print("Figures saved to:", FIG_DIR)


Figures saved to: /home/kampuser/welder_source_code/Untitled Folder/outputs/model_02_final/analysis/figures


In [60]:

# 18. 제출용 요약표 / JSON 저장

# 주요 영향변수 TOP
feature_top = feature_contrib.iloc[0]

# 가장 큰 일별 절대오차
worst_day = failure_daily.sort_values("absolute_error", ascending=False).iloc[0]

inside_row = stable_eval[stable_eval["inside_stable_candidate"] == True]
outside_row = stable_eval[stable_eval["inside_stable_candidate"] == False]

inside_rate = float(inside_row["alarm_rate"].iloc[0]) if len(inside_row) else np.nan
outside_rate = float(outside_row["alarm_rate"].iloc[0]) if len(outside_row) else np.nan

summary_table = pd.DataFrame([
    ["최종 모델", "Predictive LSTM"],
    ["Sequence length", CFG["sequence_length"]],
    ["Hidden dimension", CFG["hidden_dim"]],
    ["Best epoch", best_epoch],
    ["Threshold", BEST_THRESHOLD],
    ["Calibration Daily MAE", float(cal_daily["absolute_error"].mean())],
    ["Holdout Daily MAE", float(hold_daily["absolute_error"].mean())],
    ["Holdout 실제 불량 총량", int(hold_daily["actual_defect_count"].sum())],
    ["Holdout 모델 이상탐지 총량", int(hold_daily["predicted_anomaly_count"].sum())],
    ["이상탐지점수 주요 기여변수", feature_top["feature_kr"]],
    ["주요 변수 기여도(%)", float(feature_top["alert_contribution_pct"])],
    ["최대 일별 오차 날짜", pd.Timestamp(worst_day["date"]).strftime("%Y-%m-%d")],
    ["최대 일별 건수오차", float(worst_day["error"])],
    ["안정후보 내부 이상탐지율", inside_rate],
    ["안정후보 외부 이상탐지율", outside_rate],
], columns=["item", "value"])

summary_table.to_csv(
    ANALYSIS_DIR / "submission_summary_table.csv",
    index=False,
    encoding="utf-8-sig",
)

summary_json = {
    "model": CFG,
    "best_epoch": int(best_epoch),
    "threshold": float(BEST_THRESHOLD),
    "threshold_fit_quantile": float(BEST_Q),
    "metrics": {
        "calibration_daily_mae": float(cal_daily["absolute_error"].mean()),
        "holdout_daily_mae": float(hold_daily["absolute_error"].mean()),
        "holdout_daily_rmse": float(np.sqrt(hold_daily["squared_error"].mean())),
        "holdout_daily_bias": float(hold_daily["error"].mean()),
    },
    "holdout": {
        "rows_total": int(len(holdout_all)),
        "rows_scored": int(holdout_all["anomaly_score"].notna().sum()),
        "warmup_rows_no_prediction": int(holdout_all["anomaly_score"].isna().sum()),
        "actual_defect_total": int(hold_daily["actual_defect_count"].sum()),
        "predicted_alert_total": int(hold_daily["predicted_anomaly_count"].sum()),
    },
    "interpretation_limits": [
        "개별 생산기록 정상/불량 라벨이 없어 row-level Precision/Recall/F1을 산출할 수 없음",
        "날짜별 실제 불량 건수와 모델 이상탐지 수가 같아도 동일 제품을 탐지했다는 의미가 아님",
        "변수별 예측오차 기여도는 모델 판단 기여도이며 실제 불량의 인과적 영향도가 아님",
        "안정 공정조건은 모델 기반 저위험 후보이며 실제 무불량 보증범위가 아님",
        "working time에 세부 시각이 없어 idx/source_row를 생산순서 근사치로 사용함",
    ],
}

(ANALYSIS_DIR / "submission_summary.json").write_text(
    json.dumps(summary_json, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

display(summary_table)


,item,value
0,최종 모델,Predictive LSTM
1,Sequence length,10
2,Hidden dimension,32
3,Best epoch,80
4,Threshold,0.833912
5,Calibration Daily MAE,1.0
6,Holdout Daily MAE,0.0
7,Holdout 실제 불량 총량,11
8,Holdout 모델 이상탐지 총량,11
9,이상탐지점수 주요 기여변수,통전시간


In [61]:

# 19. 최종 출력 확인

print("=" * 72)
print("02 FINAL MODEL COMPLETE")
print("=" * 72)
print(f"최종 모델           : Predictive LSTM")
print(f"Best epoch          : {best_epoch}")
print(f"Threshold           : {BEST_THRESHOLD:.10f}")
print(f"Calibration Daily MAE: {cal_daily['absolute_error'].mean():.3f}")
print(f"Holdout Daily MAE    : {hold_daily['absolute_error'].mean():.3f}")
print(f"Holdout actual/alerts: {int(hold_daily['actual_defect_count'].sum())} / {int(hold_daily['predicted_anomaly_count'].sum())}")
print()
print("주의:")
print("- anomaly_flag는 실제 불량 확정값이 아니라 모델 기반 불량위험 이상탐지입니다.")
print("- 안정 공정조건은 모델 기반 후보범위이며 현장 품질검증이 필요합니다.")
print()
print("MODEL_DIR   :", MODEL_DIR)
print("OUTPUT_DIR  :", OUTPUT_DIR)
print("PREDICTIONS :", PRED_DIR)
print("ANALYSIS    :", ANALYSIS_DIR)
print("FIGURES     :", FIG_DIR)


02 FINAL MODEL COMPLETE
최종 모델           : Predictive LSTM
Best epoch          : 80
Threshold           : 0.8339118305
Calibration Daily MAE: 1.000
Holdout Daily MAE    : 0.000
Holdout actual/alerts: 11 / 11

주의:
- anomaly_flag는 실제 불량 확정값이 아니라 모델 기반 불량위험 이상탐지입니다.
- 안정 공정조건은 모델 기반 후보범위이며 현장 품질검증이 필요합니다.

MODEL_DIR   : /home/kampuser/welder_source_code/Untitled Folder/models
OUTPUT_DIR  : /home/kampuser/welder_source_code/Untitled Folder/outputs/model_02_final
PREDICTIONS : /home/kampuser/welder_source_code/Untitled Folder/outputs/model_02_final/predictions
ANALYSIS    : /home/kampuser/welder_source_code/Untitled Folder/outputs/model_02_final/analysis
FIGURES     : /home/kampuser/welder_source_code/Untitled Folder/outputs/model_02_final/analysis/figures
